# Rush yards walkthrough

A short notebook that **engineers features**, **fits** XGBoost and LightGBM, and **predicts** one held-out running-back game.

It uses a tiny in-memory slate (two seasons, three weeks, two RBs) so it runs without a warehouse backfill. Features are built at each game's `kickoff_at`. Training rows are only those with `kickoff_at` strictly before the game you predict.

From the repo root:

```bash
pip install -e ".[dev,notebook]"
jupyter notebook backtest/rush_yards_walkthrough.ipynb
```

This is a research demo. It calls `models.rush_yards`. It is not the production harness that reads stored `model.prediction` rows.

In [ ]:
from backtest.rush_yards_demo import (
    RB_A,
    demo_population,
    predict_player_game,
    run_walk_forward,
)
from features.types import Known, Unknown

rows = demo_population()
print(f"{len(rows)} labeled player-games")
for row in rows:
    kick = row.kickoff_at.isoformat()
    print(f"  {row.season} W{row.week} {row.gsis_id} y={row.y_true:.0f}  as_of={kick}")

## Feature engineering

Each row already went through `build_rush_rb_v1`. Same-game rushing yards are the label (`y_true`), never a feature. Missing history is `Unknown`, not 0.

In [ ]:
week3 = next(
    row for row in rows if row.gsis_id == RB_A and row.season == 2016 and row.week == 3
)
feat = week3.features
print(f"player {feat.gsis_id}  game {feat.game_id}")
print(f"as_of {feat.as_of.isoformat()}")
print()
for name in (
    "player_std_rush_yards",
    "player_last_rush_yards",
    "player_last4_rush_yards_pg",
    "player_prior_season_rush_yards_pg",
    "is_home",
    "temp",
    "depth_team",
    "report_status",
):
    value = getattr(feat, name)
    if isinstance(value, Known):
        print(f"  {name:36} Known({value.value!r})")
    elif isinstance(value, Unknown):
        print(f"  {name:36} Unknown({value.reason})")

## Walk-forward fit (2016)

For each 2016 week, fit on every game whose kickoff is before that week's earliest kickoff, then score the week. `n_estimators=40` keeps this notebook fast. The spec default is 400.

In [ ]:
result = run_walk_forward(rows, n_estimators=40)

print(f"run_id {result.run_id}")
print()
print(f"{'model':16} {'season':>8} {'MAE':>8} {'RMSE':>8} {'n':>4} {'unk':>4}")
for metric in result.metrics:
    season = "overall" if metric.season is None else str(metric.season)
    print(
        f"{metric.model_name:16} {season:>8} {metric.mae:8.2f} {metric.rmse:8.2f} "
        f"{metric.n:4d} {metric.n_unknown:4d}"
    )

## One prediction

Hold out RB_A's 2016 week 3 game. Fit both boosters on earlier kickoffs only, then predict that game and compare to the final rushing yards.

In [ ]:
pred = predict_player_game(rows, RB_A, 2016, 3, n_estimators=40)
print(f"player  {pred['gsis_id']}")
print(f"game    {pred['game_id']}")
print(f"as_of   {pred['as_of']}")
print(f"train n {pred['n_train']}")
print()
print(f"actual rushing yards   {pred['y_true']:.1f}")
print(f"XGBoost prediction     {pred['xgboost_y_hat']:.1f}")
print(f"LightGBM prediction    {pred['lightgbm_y_hat']:.1f}")